# Windstorm tracks and footprints derived from reanalysis over Europe between 1940 to present: Algorithm comparison

Production date: 2026-MM-DD

**Please note that this repository is used for development and review, so quality assessments should be considered work in progress until they are merged into the main branch.**

Produced by: Enis Gerxhalija & Olivier Burggraaff (National Physical Laboratory).

## 🌍 Use case: Use case listed here in full 

## ❓ Quality assessment question
* **How do the TempestExtremes (MSLP) and Hodges (vorticity) tracking algorithms differ in their detection of extratropical cyclones, particularly in terms of their sensitivity, duration, and intensity?**
* **How do windstorms in matchups between the TempestExtremes (MSLP) and Hodges (vorticity) tracking algorithms differ in their duration and intensity when compared to the dataset?**

**‘Context paragraph’ (no title/heading)** - a very short introduction before the assessment statement describing approach taken to answer the user question. One or two key references could be useful, if the assessment summarises literature.

**Background**

## 📢 Quality assessment statement

```{admonition} These are the key outcomes of this assessment
:class: note
* Finding 1
* Finding 2
* Finding 3
* etc
```

## 📋 Methodology

**Note:** This notebook is currently just a brain-dump in anticipation of starting the actual quality assessment at a later stage.

**Summary**  

The analysis and results are organised in the following steps, which are detailed in the sections below:

**[](section-setup)**

**[](section-download)**

**[](section-sensitivity)**

**[](section-duration)**

**[](section-intensity)**

**[](section-matchups)**

Any further notes on the method could go here (explanations, caveats or limitations).

## 📈 Analysis and results

(section-setup)=
### 1. Code setup
```{note}
This notebook uses [earthkit](https://github.com/ecmwf/earthkit) for
downloading ([earthkit-data](https://github.com/ecmwf/earthkit-data))
and visualising ([earthkit-plots](https://github.com/ecmwf/earthkit-plots)) data.
Because earthkit is in active development, some functionality may change after this notebook is published.
If any part of the code stops functioning, please raise an issue on our GitHub repository so it can be fixed.
```

#### Imports

In [ ]:
# Input / Output
from pathlib import Path
import earthkit.data as ekd
import warnings

# General data handling
import numpy as np
np.seterr(divide="ignore")  # Ignore divide-by-zero warnings
import pandas as pd
import geopandas as gpd
from functools import partial, reduce
from operator import and_ as bitwise_and
from dask.array.core import PerformanceWarning
warnings.simplefilter(action="ignore", category=PerformanceWarning)

# Visualisation
import earthkit.plots as ekp
from earthkit.plots.styles import Style
import matplotlib.pyplot as plt
plt.rcParams["grid.linestyle"] = "--"
plt.rcParams["figure.constrained_layout.use"] = True
plt.rcParams["figure.labelweight"] = "bold"
plt.rcParams["figure.titleweight"] = "bold"
from matplotlib.colors import LogNorm
from matplotlib.path import Path
from matplotlib.patches import PathPatch
from matplotlib.transforms import Affine2D
from matplotlib.lines import Line2D
import cartopy
from cartopy import crs as ccrs
import shapely
from shapely.geometry import Polygon, MultiPolygon
import cmcrameri as cmc
from tqdm import tqdm  # Progress bars
import matplotlib.colors as mcolors
# Visualisation in Jupyter book -- automatically ignored otherwise
try:
    from myst_nb import glue
except ImportError:
    glue = None

# Type hints
from typing import Callable, Iterable, Optional
from cartopy.mpl.geoaxes import GeoAxes


#### Helper functions

##### Data (pre-)processing

The following cell contains some pre-defined constants for convenience,
such as a list of variable names in the data:

In [ ]:
# Data
TRACKING_ALGORITHMS = ["hodges", "tempest_extremes"]
NUTS_LEVELS         = [0, 1, 2]

# Columns in data
SELECTION_COLUMNS   = ["algorithm", "threshold", "year",]
VARIABLES           = ["storms_number", "mean_wind_gust", "ssi", "normalised_ssi", "area_ratio", "wind_gust_ratio",]
NUTS_VARIABLES      = ["region", "country_code", "NUTS_level", "NUTS_name",]

# Organisation of data
INDEX_COLUMNS       = ["algorithm", "threshold", "region", "year",]

The following functions select data from a (Geo)DataFrame according to one or multiple conditions,
e.g. a specific year and tracking algorithm:

In [ ]:
def _select_from_column(df: pd.DataFrame, col: str, val) -> pd.Series:
    """
    Create a True/False mask for one column `col` in a (Geo)DataFrame `df`
    where the value is `val`.
    Checks for iterable/non-iterable values and switches between .isin and == accordingly.
    """
    # Check for single / multiple values
    if df[col].dtype == "O":  # String column
        if isinstance(val, str):
            # Value is str -> Single value
            iterable_val = False
        elif isinstance(val, Iterable):
            # Value is Iterable but not str -> assume Iterable[str]
            iterable_val = True
        else:
            raise ValueError(f"Cannot parse value of type `{type(val)}` for column `{col}`.")
    else:  # Non-string column, assume number or similar
        iterable_val = isinstance(val, Iterable)

    # Create True/False mask
    selection = (df[col].isin(val)) if iterable_val else (df[col] == val)
    return selection

def select_data(df: pd.DataFrame, **kwargs):
    """
    Select data from a given (Geo)DataFrame `df` according to any number of conditions,
    matching its column names.
    """
    # Create masks for all col/val pairs in kwargs, then combine with & (bitwise and)
    selection_all = [_select_from_column(df, col, val) for col, val in kwargs.items()]
    selection = reduce(bitwise_and, selection_all)

    # Apply and return
    df_selection = df[selection]
    return df_selection

The following functions allow large amounts of data to be downloaded from CDS, and to send requests based on track information.

In [ ]:
# Handling CDS size limits
def batch_requests(main_request: dict, *, batch_key: str="year", n: int=20) -> list[dict]:
    """ Take a big request (e.g. ERA5–Drought for all years) and separate it into smaller ones (size `n`). """
    from itertools import batched
    full_range = main_request[batch_key]  # e.g. [1940, 1941, ..., 2024]
    batched_range = batched(full_range, n)  # e.g. [1940, ..., 1959], [1960, ..., 1979], ...
    subrequests = [main_request | {batch_key: batch} for batch in batched_range]  # create corresponding CDS requests
    return subrequests
    
def build_era5msl_request_from_time(time_value):
    """
    Create a CDS API request for ERA5 MSLP for the given date.
    Always retrieves the 4 synoptic times.
    """
    
    # Ensure datetime
    time_value = pd.to_datetime(time_value)

    year = f"{time_value.year}"
    month = f"{time_value.month:02d}"
    day = f"{time_value.day:02d}"

    request = {
        "product_type": ["reanalysis"],
        "variable": ["mean_sea_level_pressure"],
        "year": [year],
        "month": [month],
        "day": [day],
        "time": ["00:00", "06:00", "12:00", "18:00"],
        "data_format": "netcdf",
        "download_format": "unarchived",
        "area": [70, -80, 30, 35],  # windstorm track domain
    }

    return request

##### Visualisation

In [ ]:
# Geometry / Projection
EPSG = 3035
CRS = cartopy.crs.epsg(EPSG)
MAP_KWARGS={"projection": CRS,
            "xlim": (0.25e7, 0.74e7), "ylim": (1.3e6, 5.5e6),  # Continental Europe
           }

# Define styles for land, borders, etc.
STYLE_LAND            = {"zorder": 1, "facecolor": "#ccced1", "edgecolor": "none"}
STYLE_COASTLINE       = {"zorder": 3, "edgecolor": "black", "linewidth": 1}
STYLE_NATIONAL_BORDER = {"zorder": 3, "edgecolor": "black", "linewidth": 0.5}
STYLE_CHOROPLETH      = {"zorder": 2, "edgecolor": STYLE_NATIONAL_BORDER["edgecolor"], "linewidth": STYLE_NATIONAL_BORDER["linewidth"]/5}

In [ ]:
_style_footprint = {"cmap": plt.cm.cividis, "vmin": 0, "vmax": 40}

styles = {
    "footprint": Style(**_style_footprint),
}

# Apply general settings
for style in styles.values():
    style.normalize = False

In [ ]:
def _glue_or_show(fig: plt.Figure, glue_label: Optional[str]=None) -> None:
    """
    If `glue` is available, glue the figure using the provided label.
    If not, display the figure in the notebook.
    """
    try:
        glue(glue_label, fig, display=False)
    except TypeError:
        plt.show()
    finally:
        plt.close()

def _add_textbox_to_subplots(text: str, *axs: Iterable[plt.Axes | ekp.Subplot], right=False) -> None:
    """ Add a text box to each of the specified subplots. """
    # Get the plt.Axes for each ekp.Subplot
    axs = [subplot.ax if isinstance(subplot, ekp.Subplot) else subplot for subplot in axs]

    # Set up location
    x = 0.95 if right else 0.05
    horizontalalignment = "right" if right else "left"

    # Add the text
    for ax in axs:
        ax.text(x, 0.95, text, transform=ax.transAxes,
        horizontalalignment=horizontalalignment, verticalalignment="top",
        bbox={"facecolor": "white", "edgecolor": "black", "boxstyle": "round",
              "alpha": 1})

In [ ]:
def decorate_fig(fig: ekp.Figure, *, title: Optional[str]="") -> None:
    """ Decorate an earthkit figure with land, coastlines, etc. """
    # Add progress bar because individual steps can be very slow for large plots
    with tqdm(total=4, desc="Decorating", leave=False) as progressbar:
        fig.land()
        progressbar.update()
        fig.coastlines()
        progressbar.update()
        # fig.borders()
        # progressbar.update()
        fig.gridlines(linestyle=plt.rcParams["grid.linestyle"])
        progressbar.update()
        fig.title(title)
        progressbar.update()

def decorate_fig(axs: Iterable[GeoAxes], *, title: Optional[str]="") -> None:
    """ Decorate a cartopy/matplotlib figure with land, coastlines, etc. """
    # Will be replaced by preceding (earthkit) function when earthkit-plots supports choropleth maps
    try:  # Ravel if numpy array
        axs = axs.ravel()
    except AttributeError:
        pass

    # Apply each decoration in order
    for ax in axs:
        ax.add_feature(cartopy.feature.LAND,      **STYLE_LAND)
        ax.add_feature(cartopy.feature.COASTLINE, **STYLE_COASTLINE)
        ax.add_feature(cartopy.feature.BORDERS,   **STYLE_NATIONAL_BORDER)

##### Plotting functions

In [ ]:
def plot_storm_counts(yearly_nuts_mean):
    
    # Get available thresholds
    thresholds = yearly_nuts_mean[0].index.get_level_values("threshold").unique()

    fig, axes = plt.subplots(2, 2, sharex=True)
    axes = axes.ravel()  
    
    for ax, thr in zip(axes, thresholds):
        df = yearly_nuts_mean[0]["storms_number"]
        
        # Extract time series
        hodges = df.loc["hodges", thr]
        tempest = df.loc["tempest_extremes", thr]
        
        # Plot
        hodges.plot(ax=ax, label="Hodges", linewidth=2)
        tempest.plot(ax=ax, label="TempestExtremes", linewidth=2, linestyle="--")
        
        # Styling
        ax.set_title(f"Threshold = {thr}")
        ax.set_ylabel("Storm count")
        ax.grid(True, alpha=0.3)
        ax.legend()

    # Shared x-axis label
    axes[-1].set_xlabel("Year")

    plt.suptitle("Storm Count Comparison (summed over NUTS0)", fontsize=14)
    plt.show()

def plot_nuts_greater_eq_than_spatial(nuts, column = "percentage" , label = "% years",
                       title = "% years where Δ #(Hodges - TempestExtremes) > 0",
                       vmin = 0, vmax = 100, cmap = "RdBu_r" ):

    
    chart = ekp.Map(domain=[-30, 50, 20, 85], crs=ccrs.PlateCarree())

    ax = chart.ax 

    bounds = list(range(0, 110, 10))
    norm = mcolors.BoundaryNorm(bounds, ncolors=plt.get_cmap(cmap).N)

    nuts.plot(
        column=column,
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        norm = norm,
        legend=True,
        edgecolor="black",
        vmin=vmin,          
        vmax=vmax,

        linewidth=0.3,
        legend_kwds={"label": f"{label}"},
    )

    chart.coastlines()
    chart.borders()
    ax.set_title(title)

    chart.show()

def plot_lifetime_distribution(
    data,
    labels,
    colors,
    bins=np.arange(0, 50, 1),
    sigma=1
):

    plt.figure()

    count = 0 
    
    for df, label in zip(data, labels):

        df = df.copy()

        # Lifetime (days)
        lifetime = (df["end_time"] - df["start_time"]).dt.total_seconds() / (24*60*60)

        # Number of years in the dataset
        n_years = 86

        # Histogram
        hist, bin_edges = np.histogram(lifetime, bins=bins, density=True)

        #  # per year
        hist_per_year = hist / (n_years)

        # Bin centres
        centres = (bin_edges[:-1] + bin_edges[1:]) / 2

        # Plot
        plt.plot(centres, hist_per_year, label=label, color = colors[count])

        count += 1 
    # Labels and styling
    plt.xlabel("Lifetime (days)")
    plt.ylabel("Probability density")
    plt.title("Cyclone Lifetime Distribution")
    plt.grid(True)
    plt.legend()

    plt.show()

def plot_tracks(data, id1 = None, id2 = None, tracker1="hodges", tracker2="tempest_extremes"):

    chart = ekp.Map(domain=[-80, 35, 30, 70], crs=ccrs.PlateCarree())


    
    msl_style = ekp.styles.Style(
        colors="RdBu_r",
        levels=range(980, 1030, 4),
        units="hPa",
        extend="both",
    )
    track_style = ekp.styles.Style(
        colors="magma",
        levels=range(0, 30, 2),
        units="ms-1",
        extend="max",
    )

    # --- Extract & sort data ---
    data_sorted = data.sort_index()
    
    df1, df2 = None, None

    # --- Try loading each track independently ---
    try:
        df1 = data_sorted.loc[tracker1, id1].sort_values("time").reset_index(drop=True)
    except KeyError:
        print(f"{tracker1} track not found")

    try:
        df2 = data_sorted.loc[tracker2, id2].sort_values("time").reset_index(drop=True)
    except KeyError:
        print(f"{tracker2} track not found")

    # --- MSLP background (plot first so tracks sit on top) ---
    ref_df = df1 if df1 is not None else df2
    time_str = ref_df["time"].iloc[0]

    
    request_msl = build_era5msl_request_from_time(time_str)
    
    msl_data = ekd.from_source("cds", "reanalysis-era5-single-levels", request_msl).to_xarray()
    
    msl_t = msl_data.sel(valid_time=time_str, method="nearest") / 100
    chart.contourf(msl_t, style=msl_style)
    chart.legend()  

    ax = chart.ax
    transform = ccrs.PlateCarree()
    
    # Tracker 1 (Hodges) ---
    if df1 is not None and not df1.empty:

        start1, end1 = df1.iloc[0], df1.iloc[-1]

        chart.scatter(
            df1,
            x=df1["longitude"],
            y=df1["latitude"],
            z=df1["fg10"],
            source_units="ms-1",
            style=track_style,
            s=10,
            edgecolors="blue",
            zorder=3,
        )
        chart.line(
            df1,
            x=df1["longitude"],
            y=df1["latitude"],
            c="blue",
            linestyle="dashed",
            label="Hodges",
        )

        ax.plot(start1["longitude"], start1["latitude"], "o", color="blue",
                markersize=8, zorder=5, transform=transform)
        ax.plot(end1["longitude"], end1["latitude"], "s", color="blue",
                markersize=8, zorder=5, transform=transform)


    # Tracker 2 (TempestExtremes) ---
    if df2 is not None and not df2.empty:
        start2, end2 = df2.iloc[0], df2.iloc[-1]

        chart.scatter(
            df2,
            x=df2["longitude"],
            y=df2["latitude"],
            z=df2["fg10"],
            source_units="ms-1",
            style=track_style,
            s=10,
            edgecolors="red",
            zorder=3,
        )
        chart.line(
            df2,
            x=df2["longitude"],
            y=df2["latitude"],
            c="red",
            linestyle="dotted",
            label="TempestExtremes",
        )

        ax.plot(start2["longitude"], start2["latitude"], "o", color="red",
                markersize=8, zorder=5, transform=transform)
        ax.plot(end2["longitude"], end2["latitude"], "s", color="red",
                markersize=8, zorder=5, transform=transform)


    # --- Legend (track lines + start/end markers) ---
    extra_handles = [
        Line2D([0], [0], marker="o", color="grey", linestyle="None",
               markersize=7, label="Track start"),
        Line2D([0], [0], marker="s", color="grey", linestyle="None",
               markersize=7, label="Track end"),
    ]
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles=handles + extra_handles, loc="lower left")

    # --- Map styling ---
    chart.coastlines()
    chart.gridlines()
    chart.title(f"Storm track comparison at {time_str}")
    chart.show()

##### Track analysis

In [ ]:
def get_track_ends(df):
    """
    Generate a summary dataset of cyclone tracks by extracting start and end properties.
    """
    
    records = []  # will store one summary row per track

    # Loop over each algorithm (first level of MultiIndex)
    for algo, df_algo in df.groupby(level="algorithm"):

        # Within each algorithm, loop over individual tracks (by id)
        for track_id, group in df_algo.groupby(level="id"):

            # Ensure track points are in chronological order
            group = group.sort_values("time")

            # Extract first and last points of the track
            start = group.iloc[0]
            end   = group.iloc[-1]

            # Build a summary record for this track
            records.append({
                "algorithm": algo,          # which tracking algorithm
                "id": track_id,             # track identifier

                # start point info
                "start_time": start["time"],
                "start_lat":  start["latitude"],
                "start_lon":  start["longitude"],

                # end point info
                "end_time":   end["time"],
                "end_lat":    end["latitude"],
                "end_lon":    end["longitude"],

                # intensity metric across the track
                "mean_fg10": group["fg10"].max(),  # actually max, not mean
                # "mean_fg10": group["fg10"].mean()  # alternative option
            })

    # Return everything as a DataFrame (one row per track)
    return pd.DataFrame(records)


def track_stats(tracks_start_end):

    df = tracks_start_end.copy()

    seasons = {
        "ALL": None,
        "DJF": [12, 1, 2],
        "JJA": [6, 7, 8],
    }

    # Compute duration
    df["duration_days"] = (
        df["end_time"] - df["start_time"]
    ).dt.total_seconds() / 86400

    results = {}

    for season_name, months in seasons.items():
        if months is None:
            subset = df
        else:
            subset = df[df["start_time"].dt.month.isin(months)]

        grouped = subset.groupby("algorithm")

        stats = grouped.agg(
            mean=("duration_days", "mean"),
            median=("duration_days", "median"),
            std=("duration_days", "std"),
            q25=("duration_days", lambda x: x.quantile(0.25)),
            q50=("duration_days", lambda x: x.quantile(0.5)),
            q75=("duration_days", lambda x: x.quantile(0.75)),
        )

        results[season_name] = stats

    return results


Functions used to match tracks by overlapping timestamps and mean distance between overlapping points.

In [ ]:
def match_by_time(data, TRACKING_ALGORITHMS = ["hodges", "tempest_extremes"]):
    
    df1, df2 = [data.loc[alg] for alg in TRACKING_ALGORITHMS]

    # bring `id` out of the index into a column
    df1 = df1.reset_index(level="id").rename(columns={"id": "id_hodges"})
    df2 = df2.reset_index(level="id").rename(columns={"id": "id_tempest_extremes"})

    # Find where times overlap
    time_overlap = pd.merge(df1, df2, on="time", how="inner", suffixes=[f"_{alg}" for alg in TRACKING_ALGORITHMS])
    
    overlap_counts = (
        time_overlap.groupby(["id_hodges", "id_tempest_extremes"])["time"]
        .nunique()
    ) # count number of overlapping time points.
    
    results = []
    
    for (hodges_id, tempest_id), overlap_points in overlap_counts.items(): # return key and value
    
        hodges_points = data.xs(
            ("hodges", hodges_id), level=("algorithm", "id")
        )["time"].nunique() # how many points in the tracked storm for id.
    
        tempest_points = data.xs(
            ("tempest_extremes", tempest_id), level=("algorithm", "id")
        )["time"].nunique() # how many points in the tracked storm for id.
    
        mean_points = (hodges_points + tempest_points) / 2 # mean points between paired storms.
    
        time_match = overlap_points / mean_points # number of points overlapped between paired storm.
    
        match_true = time_match > 0.6 # condition for time overlap
    
        results.append({
            "hodges_id": hodges_id,
            "tempest_id": tempest_id,
            "overlap_points": overlap_points,
            "hodges_points": hodges_points,
            "tempest_points": tempest_points,
            "time_match": time_match,
            "match": match_true
        })
    
    results_df = pd.DataFrame(results)
    
    time_matches_only = results_df[results_df["match"]] # only keep the matches

    return time_matches_only 

def match_by_mean_distance(data, time_matches_only):

    time_matches_only = time_matches_only.copy()

    hodges = data.loc["hodges"]
    tempest = data.loc["tempest_extremes"]
    
    for hodges_id, tempest_id in zip(time_matches_only["hodges_id"], time_matches_only["tempest_id"]):
    
        import haversine as hs   
        from haversine import Unit
                                
        distances = []

        # extract tracks
        h_track = hodges.loc[hodges_id].set_index("time")
        t_track = tempest.loc[tempest_id].set_index("time")

        # align on common timestamps
        common_times = h_track.index.intersection(t_track.index) # return timestamps
    
        for t in common_times:

            h = h_track.loc[t]
            te = t_track.loc[t]

            distances.append(
                hs.haversine(
                    (h["latitude"], h["longitude"]),
                    (te["latitude"], te["longitude"]),
                    unit=Unit.KILOMETERS
                )
            )

        if distances:
            time_matches_only.loc[
                (time_matches_only["hodges_id"] == hodges_id) &
                (time_matches_only["tempest_id"] == tempest_id),
                "distance"
            ] = np.mean(distances)

    return time_matches_only

Functions used to analyse sensitivity of storm indicator download.

In [ ]:
def filter_storms_per_season(data, winter_months = [12, 1, 2], summer_months = [6, 7, 8]):

    data_copy = data.copy()
    
    # Extract month
    data_copy["month"] = data_copy["time"].dt.month
    
    # Winter: Dec (12), Jan (1), Feb (2)
    winter_storms = data_copy[data_copy["month"].isin(winter_months)]
    
    # Summer: Jun (6), Jul (7), Aug (8)
    summer_storms = data_copy[data_copy["month"].isin(summer_months)]

    return winter_storms, summer_storms

def count_storms_per_month(data, algorithm):
    subset = data.loc[algorithm].copy()
    subset["month"] = subset["time"].dt.month
    subset["storm_id"] = subset.index.get_level_values("id")
    return subset.groupby("month")["storm_id"].nunique()

def compute_fractions(count, reference):
    winter_months = [12, 1, 2]
    summer_months = [6, 7, 8]
    winter = count.loc[winter_months].sum() / reference.loc[winter_months].sum()
    summer = count.loc[summer_months].sum() / reference.loc[summer_months].sum()
    count_ratio =  count.loc[summer_months].sum() / count.loc[winter_months].sum() 
    ref_ratio = reference.loc[summer_months].sum() /  reference.loc[winter_months].sum()
    return winter, summer, count_ratio, ref_ratio

Functions used to spatially analyse sensitivity of storm indicator download in NUTS region.

In [ ]:
def compute_mean_by_nuts(data, value_column, nuts_levels=(0, 1, 2)):
    """
    Compute mean values grouped by (algorithm, threshold) for each NUTS level.
    """
    # Take mean across years (frequency).
    data_grouped = (
        data[[value_column, "NUTS_level"]]
        .groupby(level=("algorithm", "threshold", "region"))
        .mean(numeric_only=True)
    )

    # Split by NUTS level
    nuts_groups = {
        level: data_grouped.loc[data_grouped["NUTS_level"] == level]
        for level in nuts_levels
    }

    # Take mean across entire domain
    result = {
        level: df.groupby(level=("algorithm", "threshold")).mean(numeric_only=True)
        for level, df in nuts_groups.items()
    }

    return result

def computer_percentage_greater_eq(data, nuts_gdf, value_column="mean_wind_gust",
    threshold=0, levels=(0, 1, 2)):
    
    
    hodges_greater = data[value_column].loc["hodges"] >= data[value_column].loc["tempest_extremes"]

    percentage = (
        hodges_greater
        .groupby(level=("threshold", "region"))
        .mean() * 100
    )

    percentage = percentage.loc[threshold].reset_index(name="percentage")
    
    map_df = nuts_gdf.merge(
        percentage,
        left_on="region",
        right_on="region",
        how="left",
    )
    
    map_df = map_df.to_crs(epsg=4326)
    
    nuts_maps = {
            level: map_df[map_df["LEVL_CODE"] == level]
            for level in levels
        }

    return nuts_maps

def compute_yearly_nuts_aggregation(
    data,
    value_column,
    nuts_levels=(0, 1, 2),
    agg_func="sum",
):
    """
    Compute yearly aggregation per NUTS level.
    """

    # Split by NUTS level
    yearly_nuts_groups = {
        level: data.loc[data["NUTS_level"] == level]
        for level in nuts_levels
    }

    # Aggregate across domain (per year)
    if agg_func == "sum":
        yearly_nuts_agg = {
            level: df.groupby(level=("algorithm", "threshold", "year")).sum(numeric_only=True)
            for level, df in yearly_nuts_groups.items()
        }
    elif agg_func == "mean":
        yearly_nuts_agg = {
            level: df.groupby(level=("algorithm", "threshold", "year")).mean(numeric_only=True)
            for level, df in yearly_nuts_groups.items()
        }
    else:
        raise ValueError("agg_func must be 'sum' or 'mean'")

    return yearly_nuts_agg

(section-download)=
### 2. Download

In [ ]:
domain_tracks = ekp.geo.domains.Domain.from_bbox([-80, 35, 5, 70], name="Europe")
domain_footprints = ekp.geo.domains.Domain.from_bbox([-25, 35, 30, 70], name="Europe")

In [ ]:
dataset = "sis-european-wind-storm-reanalysis"

#### Windstorm track download 

In [ ]:
tracking_algorithms = ["hodges", "tempestextremes"]

request_variables = {
    "variable": "all",
    "tracking_algorithm": tracking_algorithms,
}

request_time = {
    "year": [f"{year}" for year in range(1940, 2025)],
    "month": [f"{month:02}" for month in range(1, 13)],
    "day": [f"{day:02}" for day in range(1, 32)],
}

request = {
    "product": "windstorm_track",
    "event_aggregation": "single_event",
} | request_variables | request_time

track_data = ekd.from_source("cds", dataset, request)
track_data = track_data.to_pandas()
track_data["time"] = pd.to_datetime(track_data["time"])
track_data = track_data.set_index(["algorithm", "id"]) # Reindex
track_data

#### Windstorm summary indicators
We download the full windstorm summary indicator table from the
[sis-european-wind-storm-reanalysis](https://doi.org/10.24381/bf1f06a9)
CDS entry.
Because this is a simple table,
the data are loaded into a Pandas [DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html).
Geographical information will be added to this table in the next subsection.

In [ ]:
request = {
    "product": "windstorm_summary_indicator",
    # Note that we do not use the TRACKING_ALGORITHMS constant here because the CDS
    # does not have an _ in `tempestextremes` but the resulting table does.
} | request_variables

indic_data = ekd.from_source("cds", dataset, request)
indic_data = indic_data.to_pandas()
indic_data

#### Ancillary: NUTS shapefiles
The summary table is organised by NUTS
(Nomenclature of Territorial Units for Statistics)
level 0, 1, and 2
region.
To visualise these areas,
we need the corresponding shapefiles,
which can be retrieved from [GISCO](https://ec.europa.eu/eurostat/web/gisco/geodata/statistical-units/territorial-units-statistics).
We load these data into a GeoPandas [GeoDataFrame](https://geopandas.org/en/latest/docs/reference/api/geopandas.GeoDataFrame.html):

In [ ]:
# Set up GISCO URL and request
base = "https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/"
fname = "NUTS_RG_20M_2021_3035_LEVL_{level}.geojson"
# RG: Polygons
# 20M: 1:20 Million scale
# 2021: Year
# 3035: EPSG CRS (projection)
# LEVEL: Iterate over NUTS_LEVELS (0, 1, 2)
urls = [base + "/" + fname.format(level=level) for level in NUTS_LEVELS]

# Download data from GISCO
nuts_gdf = [gpd.read_file(url) for url in urls]

# Merge NUTS levels into one table
nuts_gdf = gpd.GeoDataFrame(pd.concat(nuts_gdf))
nuts_gdf = nuts_gdf.rename(columns={"NUTS_ID": "region"})

The NUTS shapes are merged into the indicator data:

In [ ]:
# Merge NUTS polygons into data
indic_data = nuts_gdf.merge(indic_data, on="region")

# Rename some columns for ease of use
indic_data = indic_data.rename(columns={"LEVL_CODE": "NUTS_level", "CNTR_CODE": "country_code", "NAME_LATN": "NUTS_name"})

# # Filter only desired columns, in pre-set order
indic_data = indic_data[[*SELECTION_COLUMNS, *NUTS_VARIABLES, *VARIABLES, "geometry",]]

# Display result
indic_data

#### Re-index indicator data
Lastly, we re-index the data by tracking algorithm, threshold, region, and year, for easier subsampling in the analysis later:

In [ ]:
# Re-index data for easier selection
indic_data = indic_data.set_index(INDEX_COLUMNS)
    
# Display result
indic_data

#### Difference between algorithms
Some of the following analysis will focus on the per-point difference between the Hodges and TempestExtremes algorithms.
Here, we calculate this difference and append it to the GeoDataFrame as a third entry along the `algorithm` index:

In [ ]:
# Setup: Copy DataFrame format
difference = indic_data.loc["hodges"].copy()

# Calculate difference
difference[VARIABLES] = indic_data.loc["hodges"][VARIABLES] - indic_data.loc["tempest_extremes"][VARIABLES]

# Append to existing DataFrame, convert back to GeoDataFrame
difference = pd.concat({"difference": difference}, names=["algorithm"])  # Add `algorithm` index level to `difference`
indic_data = pd.concat([indic_data, difference])  # Append to existing data (converts to regular DataFrame)
indic_data = gpd.GeoDataFrame(indic_data)  # Convert to GeoDataFrame again
indic_data = indic_data.loc[:, ~indic_data.columns.duplicated()] # Remove duplicate columns

# Display result
indic_data

(section-sensitivity)=
### 3. Sensitivity

#### Storm Count - from Indicators

In [ ]:
data_mean_freq_NUTS = compute_mean_by_nuts(indic_data, "storms_number")

Mean difference in annual frequency of storms in NUTS region 0 between Hodges and TempestExtremes

In [ ]:
# --- Mean Hodges - Tempest per year ---
data_mean_freq_NUTS[0].loc[["difference"]].rename(
    columns={"storms_number": "Frequency / yr"}
)

Monthly storm count taken from the track dataset with no matchups. #TODO: Thinking below is not needed if we already have data from tracks?

In [ ]:
yearly_nuts_sum = compute_yearly_nuts_aggregation(indic_data, "storms_number")

plot_storm_counts(yearly_nuts_sum)

#### Storm Count - from Tracks

Total storm count

In [ ]:
total_track_counts = (
    track_data.reset_index()
    .groupby("algorithm")["id"]
    .nunique()
)

# Define colors explicitly based on algorithm names
colors = [
    "blue" if alg == "hodges" else "red"
    for alg in total_track_counts.index
]

plt.figure(figsize=(7, 5))

plt.bar(total_track_counts.index, total_track_counts, color=colors)

# Labels and title
plt.xlabel("Algorithm", fontsize=12)
plt.ylabel("Total Storm Count", fontsize=12)
plt.title("Total Number of Tracks by Algorithm", fontsize=14)

# Improve tick appearance
plt.xticks(rotation=0)
plt.grid(axis="y", linestyle="--", alpha=0.6)

plt.ylim([0, 2000])
plt.show()


Monthly storm count

In [ ]:
tracks_start_end = get_track_ends(track_data)


# Extract month by name from start_time
tracks_start_end["month"] = tracks_start_end["start_time"].dt.month_name()

# Count number of tracks per month per algorithm
monthly_counts = (
    tracks_start_end
    .groupby(["algorithm", "month"])
    .size()
    .unstack(fill_value=0)
) 

# Reorder as by default alphabetical!
month_order = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]

monthly_counts = monthly_counts[month_order] # Reorder columns

monthly_counts.T.plot(marker="o", color=colors)

plt.ylabel("Total Number of Tracks")
plt.title("Monthly ETC Counts by Algorithm")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

Annual storm count

In [ ]:
# Extract month by name from start_time
tracks_start_end["year"] = tracks_start_end["start_time"].dt.year

# Count number of tracks per month per algorithm
yearly_counts = (
    tracks_start_end
    .groupby(["algorithm", "year"])
    .size()
    .unstack(fill_value=0)
) 

yearly_counts.T.plot(marker="o", color=colors)

plt.ylabel("Total Number of Tracks")
plt.title("Yearly ETC Counts by Algorithm")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

Seasonal storm count comparison #TODO: Make these into two tables or convert into a winter, summer, winter/summer table. (make below easier to read).

In [ ]:
tempest_count = count_storms_per_month(track_data, "tempest_extremes") 
hodges_count = count_storms_per_month(track_data, "hodges")

results = {}

results["Algo Comparison"] = compute_fractions(hodges_count, tempest_count)

fraction_table = pd.DataFrame({
    "Winter Fraction (Hodges / Tempest)": [results["Algo Comparison"][0]],
    "Summer Fraction (Hodges / Tempest)": [results["Algo Comparison"][1]],
    "Hodges (Summer / Winter)": [results["Algo Comparison"][2]],
    "Tempest (Summer / Winter)": [results["Algo Comparison"][3]],
}, index=["Fractional Value"])

fraction_table

Not sure if we still need this if above? But could be good to put some numbers to it.

#### Sensitivity in matchups

Description of the matchup algorithm and why we are doing this...

In [ ]:
time_matches_only = match_by_time(track_data)

matches_only = match_by_mean_distance(track_data, time_matches_only)

filtered_matches_only = matches_only[matches_only["distance"] < 300].dropna()
not_matched =  matches_only[matches_only["distance"] >= 300].dropna()

Below is matchup success %, brief description why we are doing this...

In [ ]:
total_matches = filtered_matches_only["match"].count()
lower_number_count = track_data.loc["tempest_extremes"].index.nunique()
percent_matches = (total_matches / lower_number_count)*100
print(f"Percentage of matches between the Hodges and Tempest Extremes = {percent_matches:.2f}%")

#### Monthly matchup % ...

In [ ]:
matches_with_monthly_name = filtered_matches_only.reset_index().copy()

for count , (hodges_id, tempest_id) in enumerate(zip(filtered_matches_only["hodges_id"], filtered_matches_only["tempest_id"])):
    h_track = track_data.loc["hodges"].loc[hodges_id]["time"]
    te_track = track_data.loc["tempest_extremes"].loc[tempest_id]["time"]

    common_times = h_track[h_track.isin(te_track)]
    first_month = common_times.iloc[0].month_name()
    
    # Extract month by name from start_time
    matches_with_monthly_name.loc[count, "month"] = first_month

In [ ]:
# Count number of tracks per month per algorithm
matched_monthly_counts = (
    matches_with_monthly_name
    .groupby(["month"])
    .size()
) 

matched_monthly_counts = matched_monthly_counts[month_order] # Reorder columns

fraction_matched_monthly = (matched_monthly_counts / monthly_counts.loc["tempest_extremes"])*100

fraction_matched_monthly.T.plot(marker="o")

plt.ylabel("% of Matched Tracks")
plt.title("Monthly ETC Matchup %")
plt.grid(True, linestyle="--", alpha=0.6)
plt.ylim([0, 100])
plt.show()

#### Seasonal matchup % ...

In [ ]:
winter_months = ["December", "January", "February"]
summer_months = ["June", "July", "August"]

winter_matchup_count = matched_monthly_counts[winter_months].sum()
winter_tempest_count = tempest_monthly_counts[winter_months].sum()
winter_hodges_count = hodges_monthly_counts[winter_months].sum()

summer_matchup_count = matched_monthly_counts[summer_months].sum()
summer_tempest_count = tempest_monthly_counts[summer_months].sum()
summer_hodges_count = hodges_monthly_counts[summer_months].sum()


winter_matchup_frac = round((winter_matchup_count / winter_tempest_count)*100)
summer_matchup_frac = round((summer_matchup_count/ summer_tempest_count)*100)

In [ ]:
matchup_table = pd.DataFrame(
    {
        "JJA (#)": ["×100", summer_hodges_count, summer_tempest_count],
        "Hodges (%)": [winter_hodges_count, 100, summer_matchup_frac],
        "Tempest (%)": [winter_tempest_count, winter_matchup_frac, 100],
    },
    index=["DJF (#)", "Hodges (%)", "Tempest (%)"]
)

matchup_table.index.name = "Method"

In [ ]:
def style_table(val):
    if val == 100:
        return "background-color: #bfbfbf; font-weight: bold;"   # grey (diagonal)
    elif val == "×100":
        return "background-color: #d9d9d9;"  # lighter grey
    elif val in [winter_hodges_count, winter_tempest_count]:
        return "color: blue; background-color: #d9d9d9"  # blue
    elif val in [summer_hodges_count, summer_tempest_count]:
        return "color: red; background-color: #d9d9d9"  # red
    elif val in [summer_matchup_frac]:
        return "background-color: #F4D7CC"  # red
    elif val in [winter_matchup_frac]:
        return "background-color:  #D9E2F3"  # blue background
    else:
        return ""

styled = matchup_table.style.applymap(style_table)
styled


(section-duration)=
### 4. Duration

#### Quantiles for duration of storms (no matchups)

In [ ]:
tracks_start_end = get_track_ends(track_data)

stats = track_stats(tracks_start_end)

In [ ]:
stats["ALL"]

In [ ]:
stats["DJF"]

In [ ]:
stats["JJA"]

#### Lifetime distribution (no matchups)

In [ ]:
tracks_start_end = get_track_ends(track_data)
hodges_track_start_end = tracks_start_end[tracks_start_end["algorithm"] == "hodges"]
tempest_track_start_end = tracks_start_end[tracks_start_end["algorithm"] == "tempest_extremes"]

In [ ]:
plot_lifetime_distribution(
    data=[hodges_track_start_end, tempest_track_start_end],
    labels=["hodges", "tempestextremes"],
    colors = colors,
    bins=np.arange(0, 20, 0.5),
)

#### Quantiles for duration of storms (matchups only)

In [ ]:
h_track = tracks_start_end[tracks_start_end["algorithm"] == "hodges"]
te_track = tracks_start_end[tracks_start_end["algorithm"] == "tempest_extremes"]

In [ ]:
filtered_matches_only

In [ ]:
h_ids_to_keep = filtered_matches_only["hodges_id"]  # or whatever your Series is

h_filtered_tracks = h_track[
    h_track["id"].isin(h_ids_to_keep)
]
h_filtered_tracks

In [ ]:
te_ids_to_keep = filtered_matches_only["tempest_id"]

te_filtered_tracks = te_track[
    te_track["id"].isin(te_ids_to_keep)
]
te_filtered_tracks

In [ ]:
plot_lifetime_distribution(
    data=[h_filtered_tracks, te_filtered_tracks],
    labels=["hodges", "tempestextremes"],
    colors = colors,
    bins=np.arange(0, 20, 0.5),
)

(section-intensity)=
### 5. Intensity

#### Mean wind gust - from Indicators

In [ ]:
data_mean_wind_NUTS = compute_mean_by_nuts(indic_data, "mean_wind_gust")

# --- Mean Hodges - Tempest per year ---
data_mean_wind_NUTS[0].loc[["difference"]]

In [ ]:
threshold = 0.0

nuts_percentage = computer_percentage_greater_eq(indic_data, nuts_gdf, value_column = "mean_wind_gust", threshold = threshold)

plot_nuts_greater_eq_than_spatial(nuts_percentage[0],
                   title = f"% Years with Δ Mean Wind Gust(Wind Gust > Threshold = {threshold}) (H − TE) >= 0",)

In [ ]:
threshold = 25.0

nuts_percentage = computer_percentage_greater_eq(indic_data, nuts_gdf, value_column = "mean_wind_gust", threshold = threshold)

plot_nuts_greater_eq_than_spatial(nuts_percentage[0],
                   title = f"% Years with Δ Mean Wind Gust(Wind Gust > Threshold = {threshold}) (H − TE) >= 0",)

#### Normalised Storm Severity Index (NSSI) - from Indicators

In [ ]:
data_nssi_NUTS = compute_mean_by_nuts(indic_data, "normalised_ssi")

# --- Mean Hodges - Tempest per year ---
data_nssi_NUTS[0].loc[["difference"]]

In [ ]:
threshold = 0.0

nuts_percentage = computer_percentage_greater_eq(indic_data, nuts_gdf, value_column = "normalised_ssi", threshold = threshold)

plot_nuts_greater_eq_than_spatial(nuts_percentage[0],
                   title = f"% Years with Δ NSSI(Wind Gust > Threshold = {threshold}) (H − TE) >= 0",)

(section-matchups)=
### 6. Matchups

#### Comparison between matchups

In [ ]:
h_start_end_track = tracks_start_end[tracks_start_end["algorithm"] == "hodges"]
te_start_end__track = tracks_start_end[tracks_start_end["algorithm"] == "tempest_extremes"]
h_track = track_data.loc["hodges"]
te_track = track_data.loc["tempest_extremes"]

comparison = []
for hodges_id, tempest_id in zip(filtered_matches_only["hodges_id"], filtered_matches_only["tempest_id"]):

    h = h_start_end_track[h_start_end_track["id"] == hodges_id].iloc[0]
    te = te_start_end__track[te_start_end__track["id"] == tempest_id].iloc[0]

    comparison.append({
    
                # time differences (Timedelta)
                "start_time_diff": h["start_time"] - te["start_time"],
                "end_time_diff": h["end_time"] - te["end_time"],
    
                # spatial differences (just raw differences)
                "start_lat_diff": h["start_lat"] - te["start_lat"],
                "start_lon_diff": h["start_lon"] - te["start_lon"],
                "end_lat_diff": h["end_lat"] - te["end_lat"],
                "end_lon_diff": h["end_lon"] - te["end_lon"],

                # intensity differences (just raw max wind speed along TOTAL track)
                "hodges_mean_max_fg10": h_track["fg10"].loc[hodges_id].max(),
                "tempest_mean_max_fg10": te_track["fg10"].loc[tempest_id].max(),
            })
comparison_df = pd.DataFrame(comparison)

In [ ]:
comparison_df.mean()

Mean max wind gust for Hodges

In [ ]:
h_track = track_data.loc["hodges"]
te_track = track_data.loc["tempest_extremes"]

In [ ]:
h_mean_of_max = h_track.groupby(level=0)['fg10'].max().mean()
h_mean_of_max

Mean max wind gust for Tempest

In [ ]:
te_mean_of_max = te_track.groupby(level=0)['fg10'].max().mean()
te_mean_of_max

#### Visualising matchups

All matchups that satisfy 60%  timestamp overlap and mean distance is less than 300km for overlapping points.

In [ ]:
for hodges_id, tempest_id in zip(filtered_matches_only["hodges_id"], filtered_matches_only["tempest_id"]):
    plot_tracks(track_data, hodges_id, tempest_id) # plot each matchup with MSLP at start time for earliest storm laid on top.

All matchups that satisfy 60%  timestamp overlap but do not satisfy mean distance is less than 300km for overlapping points.

In [ ]:
for hodges_id, tempest_id in zip(filtered_not_matched["hodges_id"], filtered_not_matched["tempest_id"]):
    plot_tracks(track_data, hodges_id, tempest_id) # plot each matchup with MSLP at start time for earliest storm laid on top.

## ℹ️ If you want to know more

### Key resources

Code libraries used:
* [GeoPandas](https://geopandas.org/en/stable/)
* [earthkit](https://github.com/ecmwf/earthkit)
  * [earthkit-data](https://github.com/ecmwf/earthkit-data)
  * [earthkit-plots](https://github.com/ecmwf/earthkit-plots)

### References
